# berts-gg-qa-submit

Submit-only notebook (internet OFF).

- Load 3 trained SWA model weights from `/kaggle/working/ggqa_bert_models`
- Infer test predictions model-by-model
- Average ensemble + clipping + snapping
- Save `submission.csv`


In [ ]:
import os, re, json, html
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel


In [ ]:
OUT_DIR = '/kaggle/working/ggqa_bert_models'
ART_DIR = f'{OUT_DIR}/artifacts'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

MODEL_ZOO = {
    'roberta_base': '/kaggle/input/datasets/abhishek/roberta-base',
    'roberta_large': '/kaggle/input/datasets/marshal02/robertalarge',
    'deberta_v3_base': '/kaggle/input/datasets/debarshichanda/debertav3base'
}

with open(f'{ART_DIR}/target_cols.json', 'r') as f:
    target_cols = json.load(f)
with open(f'{ART_DIR}/label_values.json', 'r') as f:
    label_values = json.load(f)
with open(f'{ART_DIR}/clippings.json', 'r') as f:
    CLIPPINGS = {k: tuple(v) for k, v in json.load(f).items()}
with open(f'{ART_DIR}/model_manifest.json', 'r') as f:
    model_manifest = json.load(f)

print('models:', model_manifest)


In [ ]:
puncts = [',', '.', '"', ':', ')', '(', '-', '!', '?', '|', ';', "'", '$', '&', '/', '[', ']', '>', '%', '=', '#', '*', '+', '\\', '\\n', '\\t']
mispell_dict = {"can't":"cannot","won't":"will not","don't":"do not","doesn't":"does not","didn't":"did not","it's":"it is","i'm":"i am","you're":"you are","they're":"they are"}

def clean_text(x):
    x = str(x)
    for punct in puncts:
        x = x.replace(punct, f' {punct} ')
    return x

def clean_numbers(x):
    x = re.sub('[0-9]{5,}', '#####', x)
    x = re.sub('[0-9]{4}', '####', x)
    x = re.sub('[0-9]{3}', '###', x)
    x = re.sub('[0-9]{2}', '##', x)
    return x

def replace_typical_misspell(text):
    text = str(text)
    for k, v in mispell_dict.items():
        text = text.replace(k, v)
    return text

def clean_data(df, columns):
    out = df.copy()
    for c in columns:
        out[c] = out[c].fillna('').apply(lambda x: html.unescape(str(x)))
        out[c] = out[c].apply(clean_numbers).str.lower().apply(clean_text).apply(replace_typical_misspell)
    return out


In [ ]:
test = pd.read_csv('/kaggle/input/competitions/google-quest-challenge/test.csv')
sample = pd.read_csv('/kaggle/input/competitions/google-quest-challenge/sample_submission.csv')
test = clean_data(test, ['question_title','question_body','answer'])


In [ ]:
class QADataset(Dataset):
    def __init__(self, df, tokenizer, max_len_q=256, max_len_a=256):
        self.df = df.reset_index(drop=True)
        self.tok = tokenizer
        self.max_len_q = max_len_q
        self.max_len_a = max_len_a

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        qtxt = f"{r['question_title']} [SEP] {r['question_body']}"
        atxt = f"{r['question_title']} [SEP] {r['answer']}"
        q = self.tok(qtxt, truncation=True, max_length=self.max_len_q, padding='max_length', return_tensors='pt')
        a = self.tok(atxt, truncation=True, max_length=self.max_len_a, padding='max_length', return_tensors='pt')
        return {'q_input_ids': q['input_ids'][0], 'q_attention_mask': q['attention_mask'][0], 'a_input_ids': a['input_ids'][0], 'a_attention_mask': a['attention_mask'][0]}

class DualTowerRegressor(nn.Module):
    def __init__(self, model_path, n_targets):
        super().__init__()
        self.q_encoder = AutoModel.from_pretrained(model_path, local_files_only=True)
        self.a_encoder = AutoModel.from_pretrained(model_path, local_files_only=True)
        hidden = self.q_encoder.config.hidden_size
        self.drop = nn.Dropout(0.2)
        self.head = nn.Sequential(nn.Linear(hidden * 2, hidden), nn.GELU(), nn.Linear(hidden, n_targets))

    def encode(self, encoder, ids, mask):
        out = encoder(ids, attention_mask=mask).last_hidden_state
        m = mask.unsqueeze(-1)
        return (out * m).sum(1) / m.sum(1).clamp(min=1)

    def forward(self, q_ids, q_mask, a_ids, a_mask):
        qv = self.encode(self.q_encoder, q_ids, q_mask)
        av = self.encode(self.a_encoder, a_ids, a_mask)
        return self.head(self.drop(torch.cat([qv, av], 1)))

def fix_state_dict(state_dict):
    return {(k[7:] if k.startswith('module.') else k): v for k, v in state_dict.items()}

def predict_model(model, loader):
    model.eval()
    outs=[]
    with torch.no_grad():
        for b in loader:
            p = model(
                b['q_input_ids'].to(device), b['q_attention_mask'].to(device),
                b['a_input_ids'].to(device), b['a_attention_mask'].to(device)
            ).detach().cpu().numpy()
            outs.append(p)
    return np.concatenate(outs, axis=0)


In [ ]:
def apply_clipping(preds, target_cols):
    z = preds.copy()
    for j,c in enumerate(target_cols):
        if c in CLIPPINGS:
            lo,hi = CLIPPINGS[c]
            z[:,j] = np.clip(z[:,j], lo, hi)
    return z

def snap_to_known_labels(preds, target_cols, label_values):
    out = np.zeros_like(preds)
    for j,c in enumerate(target_cols):
        vals = np.asarray(label_values[c], dtype=np.float32)
        idx = np.abs(preds[:, [j]] - vals.reshape(1, -1)).argmin(axis=1)
        out[:,j] = vals[idx]
    return out

def ensure_not_constant(preds):
    out = preds.copy()
    for j in range(out.shape[1]):
        if np.allclose(out[:,j], out[0,j]):
            out[:,j] = out[:,j] + np.linspace(0, 1e-7, out.shape[0], dtype=np.float32)
    return out


In [ ]:
test_preds = []
for model_name, model_path in MODEL_ZOO.items():
    print('loading', model_name)
    tok = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
    ds = QADataset(test, tok, 256, 256)
    ld = DataLoader(ds, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

    model = DualTowerRegressor(model_path, len(target_cols)).to(device)
    sd = torch.load(model_manifest[model_name], map_location=device)
    model.load_state_dict(fix_state_dict(sd), strict=False)

    p = predict_model(model, ld)
    test_preds.append(p)

ensemble_test = np.mean(np.stack(test_preds, axis=0), axis=0)
final_test = snap_to_known_labels(apply_clipping(ensemble_test, target_cols), target_cols, label_values)
final_test = ensure_not_constant(final_test)

sub = sample.copy()
sub[target_cols] = final_test
sub.to_csv('submission.csv', index=False)
print('saved submission.csv', sub.shape)
sub.head()
